# Week 1 — Research Question and Provisional Lane

Setup phase, Week 1. Before touching any model, I want to nail down what question I'm
actually answering and why it's worth answering.


## 1. My lane and why

**Going with Lane 2: Refresh / Content Opportunity Scoring.**

The question is basically: *which pages should someone look at first if they only have time
to review a handful this week?*

I picked this over the other three lanes for a pretty simple reason — it's the one the
starter dataset is already set up for. `content_refresh_anonymized.csv` has
`trend_direction`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr` — all
the stuff this lane needs, no extra joins or hunting around required. It also comes with a
baseline already sketched out (the `stale_visible_page` / `declining_with_demand` type rules
from the lane guide), so I have something honest to compare a model against instead of just
guessing at whether it's any good.

This is still provisional — I can change my mind up to Week 4. If the label turns out to be
too messy once I dig in, I'll probably fall back to Lane 1 (Ranking Signal Analysis) instead.


## 2. The question

**Unit of analysis:** one page. Not a client, not a day — a page, because that's the thing
someone would actually open and edit.

**Question:** among pages that already get some search traffic, which ones are the best
candidates to review this week for a refresh, an expansion, or just protecting what's working?

**Output:** a ranked list — score, a short reason why it's flagged, and roughly how confident
I am. Not a single yes/no.

**The action:** an editor takes the top N pages (however many the team can actually get
through) and either refreshes, expands, fixes the title/meta, or just leaves it and monitors.

**What a wrong call costs:**
- Flag a page that didn't need it → wasted review time. Annoying, not a big deal.
- Miss a page that's actually declining → it keeps quietly losing traffic until someone
  notices by accident. That's the more expensive mistake, so I care more about not missing
  real problems than about being perfectly precise everywhere.

**Why this needs data/ML and not just a rule:** the starter pipeline already has a hand-built
rule for this (fixed weights like 0.40 / 0.30 / 0.25 / 0.05 on a few signals), but those
weights were never actually checked against what happened afterward. A model can learn which
signals really line up with outcomes instead of using someone's best guess. The harder part
isn't fitting the model though — it's figuring out what "declining" should even mean, making
sure I'm not leaking the answer into the features, and checking the model isn't just
memorizing one client's pages.


## 3. Quick look at the data

Loading the starter CSV and applying the same filter the pipeline uses
(`impressions_90d > 0`, `content_age_days >= 90`, one row per page).

The setup cell below just makes sure we're standing in the right folder — it clones the repo
if this is running on Colab, or steps up from `work/notebooks/` if running locally.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks/ back to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")


Working dir: /content/Flyrank-ML-Internship
Starter data found.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

eligible = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates("content_id")
)

print(f"Total rows: {len(df):,}")
print(f"Eligible pages: {len(eligible):,}")


Total rows: 30,000
Eligible pages: 30,000


In [3]:
decline_rate = (eligible["trend_direction"] == "down").mean()
print(f"Pages currently trending down: {decline_rate:.1%}")

decline_with_demand = (
    (eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)
).mean()
print(f"Declining but still getting real traffic (>=100 impressions/90d): {decline_with_demand:.1%}")

print(f"Median impressions_90d: {eligible['impressions_90d'].median():.0f}")
print(f"Median clicks_90d: {eligible['clicks_90d'].median():.0f}")


Pages currently trending down: 54.2%
Declining but still getting real traffic (>=100 impressions/90d): 43.8%
Median impressions_90d: 731
Median clicks_90d: 1


**What this tells me:** about 54% of eligible pages are marked as declining, and 44%
of them are declining while still pulling in real traffic — so it's not just dead pages with
zero visitors. There's a real pool worth sorting through, not just filtering out. The gap
between median impressions (731) and median clicks (1) is pretty striking too — a lot of
pages get seen and basically nobody clicks. That lines up with CTR being a real, ongoing
issue across the inventory, not a one-off.

Basically: there's too much here to just eyeball, and the split between "declining" and
"actually worth fixing" isn't obvious from a single number. That's the actual problem this
lane is for.


## 4. What I can and can't say

I can talk about patterns in this data — it's real, observed, historical stuff. A ranked
list is decision support: it tells someone where to look first, it doesn't promise a page
will recover if they edit it.

I can't say I've found a Google ranking factor — I only have search/engagement signals, not
the algorithm. I can't say a refresh *causes* recovery either, unless I actually run something
like a before/after test with a control group, which this data doesn't give me.

`trend_direction` is also not a perfect label — it's based on the current window, not a
checked future outcome, and it won't tell the difference between real decline, a related page
just absorbing the traffic, or normal seasonal dips. That's something I need to dig into
before Week 2–3, not something I'm assuming is already solved.

No client names, domains, URLs, or raw queries anywhere here — just the pseudonymous IDs for
grouping.


## 5. Self-check

- Picked a lane (Refresh / Content Opportunity Scoring) and said why, not just went with the
  first option.
- Named the unit of analysis, the output, the action, and what a wrong call costs.
- Backed it up with real numbers computed live from the starter data, not copy-pasted from
  the outputs folder.
- Explained why this isn't just "train a model" — the real work is defining the label
  properly and not leaking the answer into the features.
- Kept the language careful — no causal claims, no algorithm claims.
- Nothing client-identifying anywhere.

**Still open for Week 2:** is `trend_direction == "down"` good enough to start with, or should
I move to a proper prior-90-days → next-30-days label sooner? Starting simple for now and
revisiting once I've run the baseline.
